# QC threshold check

Share of cells each method loses under different transcript and volume cutoffs, mirroring `merge_adata.py`: manual outlier regions, then volume outliers (>3× median or below the minimum volume), then transcripts and genes (≥5). Volume medians are computed over the selected samples only, not the whole cohort.

In [ ]:
SAMPLES = ["aging_s1_r1", "aging_s12_r0", "ABCAtlas_s1_r0"]
MIN_COUNTS, MIN_VOLUME = 20, 100

import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import yaml
from cellseg_benchmark import BASE_PATH
from cellseg_benchmark._constants import method_colors, method_names
from cellseg_benchmark.adata_utils import filter_spatial_outlier_cells, plot_spatial_multiplot

meta = yaml.safe_load(open(Path(BASE_PATH) / "misc" / "sample_metadata.yaml"))
cells = []
for s in SAMPLES:
    for f in sorted((Path(BASE_PATH) / "samples" / s / "sdata_z3.zarr" / "tables").glob("adata_*")):
        a, m = ad.read_zarr(f), f.name.removeprefix("adata_")
        xy = np.asarray(a.obsm.get("spatial_microns", a.obsm["spatial"]))
        cells.append(pd.DataFrame({
            "sample": s, "method": m,
            "n_counts": np.asarray(a.X.sum(axis=1)).ravel(),
            "n_genes": np.asarray((a.X != 0).sum(axis=1)).ravel(),
            "volume": a.obs["volume_final"].to_numpy() if "volume_final" in a.obs else np.nan,
            "x": xy[:, 0], "y": xy[:, 1],
        }))
        del a
cells = pd.concat(cells, ignore_index=True).astype({"sample": "category", "method": "category"})

q = ad.AnnData(obs=pd.DataFrame({"sample": cells["sample"].to_numpy()}, index=cells.index.astype(str)))
q.obsm["spatial"] = cells[["x", "y"]].to_numpy()
with tempfile.TemporaryDirectory() as tmp:
    cells["outlier"] = filter_spatial_outlier_cells(q, str(BASE_PATH), meta, tmp, remove_outliers=False).obs["spatial_outlier"].to_numpy()

def dropped(min_counts, min_volume):
    low = (cells.n_counts < min_counts) | (cells.n_genes < 5)
    big = cells.volume > 3 * cells.volume.where(~low).groupby(cells.method, observed=True).transform("median")
    return pd.DataFrame({"outlier": cells.outlier, "volume": big | (cells.volume < min_volume), "counts": low})

cells.groupby("sample").method.nunique()

In [ ]:
current = np.where(cells.method.astype(str).str.startswith("vpt_3D"), 10, 25)
scenarios = {"current": dropped(current, 10)} | {f"{c} / {v} μm³": dropped(c, v) for c in (10, 20, 25) for v in (10, 100)}
pct = pd.DataFrame({k: d.any(axis=1) for k, d in scenarios.items()}).groupby([cells.method, cells["sample"]], observed=True).mean().groupby("method", observed=True).mean() * 100
order = [m for m in method_colors if m in pct.index] + sorted(set(pct.index) - set(method_colors))
pct.loc[order].rename(index=method_names).round(1)

In [ ]:
d = scenarios[f"{MIN_COUNTS} / {MIN_VOLUME} μm³"]
labels = ["outlier region", f"volume (<{MIN_VOLUME} μm³ or >3× median)", f"<{MIN_COUNTS} counts or <5 genes"]
cells["qc"] = np.select([d.outlier, d.volume, d.counts], labels, "kept")
palette = dict(zip(["kept"] + labels, ["lightgrey", "black", "tab:blue", "tab:orange"]))
for s in SAMPLES:
    c = cells[cells["sample"] == s]
    q = ad.AnnData(obs=pd.DataFrame({"sample": c.method.astype(str).map(lambda m: method_names.get(m, m)).to_numpy(),
                                     "qc": c.qc.to_numpy()}, index=c.index.astype(str)).astype("category"))
    q.obsm["spatial"] = c[["x", "y"]].to_numpy()
    plot_spatial_multiplot(q, "qc", title=s, n_cols=6, figsize_per_ax=3, palette=palette, max_points_per_sample=50_000)